In [ ]:
# =====================================
# 0. IMPORTS
# =====================================
import os
import pandas as pd
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import torch.nn.functional as F
from sklearn.model_selection import train_test_split

import torchvision.transforms as transforms
import torchvision.models as models

# =====================================
# 1. PATHS
# =====================================
CSV_PATH = "/Users/emilymoore/Downloads/DS 4002 Project 3/cleaned_breast_lesions.csv"
IMAGE_DIR = "/Users/emilymoore/Downloads/DS 4002 Project 3/Breast Lesions USG Images/"
MASK_DIR = "/Users/emilymoore/Downloads/DS 4002 Project 3/Breast Lesions USG Masks/"

# =====================================
# 2. LOAD DATA
# =====================================
df = pd.read_csv(CSV_PATH)

df["Classification"] = df["Classification"].str.lower()
df = df.dropna(subset=["Image_filename", "Mask_tumor_filename", "Classification"])

# 70/30 split
train_df, test_df = train_test_split(
    df, test_size=0.3, stratify=df["Classification"], random_state=42
)

# =====================================
# 3. FIND MINORITY CLASS
# =====================================
class_counts = train_df["Classification"].value_counts()
minority_class = class_counts.idxmin()

print("Class distribution:\n", class_counts)
print("Minority class:", minority_class)

# =====================================
# 4. TRANSFORMS
# =====================================
base_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

augment_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor()
])

# =====================================
# 5. DATASET
# =====================================
class SegmentationDataset(Dataset):
    def __init__(self, df, image_dir, mask_dir, base_transform, augment_transform=None, minority_class=None):
        self.df = df.reset_index(drop=True)
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.base_transform = base_transform
        self.augment_transform = augment_transform
        self.minority_class = minority_class

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img_path = os.path.join(self.image_dir, str(row["Image_filename"]))
        mask_path = os.path.join(self.mask_dir, str(row["Mask_tumor_filename"]))

        image = Image.open(img_path).convert("RGB")
        mask = Image.open(mask_path).convert("L")

        # Apply augmentation ONLY to minority class
        if self.augment_transform and row["Classification"] == self.minority_class:
            image = self.augment_transform(image)
            mask = self.augment_transform(mask)
        else:
            image = self.base_transform(image)
            mask = self.base_transform(mask)

        # Ensure binary mask
        mask = (mask > 0).float()

        return image, mask, row["Classification"]

# =====================================
# 6. DATALOADERS
# =====================================
train_dataset = SegmentationDataset(
    train_df, IMAGE_DIR, MASK_DIR,
    base_transform, augment_transform, minority_class
)

test_dataset = SegmentationDataset(
    test_df, IMAGE_DIR, MASK_DIR,
    base_transform, None, None
)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=16, shuffle=False)

# =====================================
# 7. U-NET MODEL
# =====================================
class UNet(nn.Module):
    def __init__(self):
        super().__init__()

        self.encoder = models.resnet18(weights="IMAGENET1K_V1")
        self.encoder = nn.Sequential(*list(self.encoder.children())[:-2])

        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(512, 256, 2, stride=2),
            nn.ReLU(),
            nn.ConvTranspose2d(256, 128, 2, stride=2),
            nn.ReLU(),
            nn.ConvTranspose2d(128, 1, 2, stride=2),
            nn.Sigmoid()
        )

    def forward(self, x):
        x = self.encoder(x)
        x = self.decoder(x)
        return x

# =====================================
# 8. DICE LOSS (FIXED)
# =====================================
def dice_loss(pred, target, smooth=1e-6):
    pred = pred.float()
    target = target.float()

    if pred.shape != target.shape:
        pred = F.interpolate(pred, size=target.shape[2:], mode='bilinear', align_corners=False)

    pred = pred.reshape(pred.size(0), -1)
    target = target.reshape(target.size(0), -1)

    intersection = (pred * target).sum(dim=1)
    dice = (2. * intersection + smooth) / (pred.sum(dim=1) + target.sum(dim=1) + smooth)

    return 1 - dice.mean()

# =====================================
# 9. TRAINING
# =====================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = UNet().to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-4)

EPOCHS = 15

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for images, masks, _ in train_loader:
        images = images.to(device)
        masks = masks.to(device).float()

        preds = model(images)

        loss = dice_loss(preds, masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

# =====================================
# 10. DICE COEFFICIENT (EVAL ONLY)
# =====================================
def dice_coeff(pred, target, smooth=1e-6):
    if pred.dim() == 3:
        pred = pred.unsqueeze(0)
    if target.dim() == 3:
        target = target.unsqueeze(0)

    if pred.shape != target.shape:
        pred = F.interpolate(pred, size=target.shape[2:], mode='bilinear', align_corners=False)

    pred = (pred > 0.5).float()

    pred = pred.reshape(-1)
    target = target.reshape(-1)

    intersection = (pred * target).sum()
    return (2. * intersection + smooth) / (pred.sum() + target.sum() + smooth)

# =====================================
# 11. EVALUATION BY LESION TYPE
# =====================================
model.eval()

dice_scores = {"benign": [], "malignant": [], "normal": []}

with torch.no_grad():
    for images, masks, labels in test_loader:
        images = images.to(device)
        masks = masks.to(device)

        preds = model(images)

        for i in range(len(labels)):
            d = dice_coeff(
                preds[i].unsqueeze(0),
                masks[i].unsqueeze(0)
            ).item()

            dice_scores[labels[i]].append(d)

# =====================================
# 12. RESULTS
# =====================================
for lesion_type, scores in dice_scores.items():
    if len(scores) > 0:
        print(f"{lesion_type.upper()} Dice: {np.mean(scores):.4f}")